<a href="https://colab.research.google.com/github/tranduchoa518-lang/fake-news-detection/blob/main/notebooks/03_word2vec_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gensim pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 120.3 MB/s  0:00:00


In [2]:
import pandas as pd
import numpy as np

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split

print("Đã import thư viện thành công!")

Đã import thư viện thành công!


In [3]:

from google.colab import files
import pandas as pd

# Tải dữ liệu đã tiền xử lý từ Notebook 02
uploaded = files.upload()

# Lấy tên file vừa tải lên
filename = next(iter(uploaded))

# Đọc dữ liệu
df = pd.read_csv(filename)

# Kiểm tra các cột cần thiết
required_columns = ["clean_text", "label"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"File dữ liệu bị thiếu cột: {col}")

# Loại bỏ dòng thiếu dữ liệu
df = df.dropna(subset=["clean_text", "label"])

# Chuyển nội dung văn bản về dạng chuỗi
df["clean_text"] = df["clean_text"].astype(str)

print("Đã đọc dữ liệu thành công!")
print("Tên file:", filename)
print("Kích thước dữ liệu:", df.shape)

print("\nSố lượng nhãn:")
print(df["label"].value_counts())

display(df.head())


Saving preprocessed_fake_news(1).csv to preprocessed_fake_news(1).csv
Đã đọc dữ liệu thành công!
Tên file: preprocessed_fake_news(1).csv
Kích thước dữ liệu: (300, 7)

Số lượng nhãn:
label
1    150
0    150
Name: count, dtype: int64


,title,text,subject,date,label,text_length,clean_text
0,CẢNH BÁO NÓNG: Chấn động: Uống nước chanh pha ...,Một bài viết lan truyền chóng mặt trên mạng xã...,healthNews,"July 23, 2026",1,377,mt bi vit lan truyn chng mt trn mng x hi tuyn ...
1,TIN ĐỒN THẤT THIỆT: Tin khẩn: Ngân hàng X sắp ...,Nhiều tài khoản ẩn danh đồng loạt đăng tải tin...,financeNews,"October 04, 2026",1,381,nhiu ti khon n danh ng lot ng ti tin n tht thi...
2,"KHẨN CẤP: Bắt đầu từ tháng tới, người dân đi x...",Trên các hội nhóm mạng xã hội xuất hiện thông ...,socialNews,"February 04, 2025",1,341,trn cc hi nhm mng x hi xut hin thng tin ct ghp...
3,CẢNH BÁO NÓNG: Lộ diện dự án đầu tư AI siêu lợ...,Một đường dây lừa đảo sử dụng hình ảnh cắt ghé...,technologyNews,"November 17, 2025",1,352,mt ng dy la dng hnh nh ct ghp ca cc lnh cp cao...
4,SỐC: Bộ Giáo dục thông báo hủy bỏ toàn bộ kỳ t...,Một trang fanpage giả mạo đăng tải văn bản có ...,educationNews,"June 03, 2026",1,333,mt trang fanpag gi mo ng ti vn bn c con du ct ...


In [4]:
sentences = [
    text.split()
    for text in df["clean_text"].astype(str)
]

print("Số lượng văn bản:", len(sentences))

print("\nVí dụ:")
print(sentences[0][:20])

Số lượng văn bản: 300

Ví dụ:
['mt', 'bi', 'vit', 'lan', 'truyn', 'chng', 'mt', 'trn', 'mng', 'x', 'hi', 'tuyn', 'b', 'rng', 'ch', 'cn', 'ung', 'hn', 'hp', 'nc']


In [5]:
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

print("Huấn luyện Word2Vec hoàn tất!")

Huấn luyện Word2Vec hoàn tất!


In [6]:
word = "government"

if word in word2vec_model.wv:
    print("Vector của từ:", word)
    print(word2vec_model.wv[word])
else:
    print("Từ không có trong từ điển Word2Vec.")

Từ không có trong từ điển Word2Vec.


In [7]:
word = "government"

if word in word2vec_model.wv:
    similar_words = word2vec_model.wv.most_similar(
        word,
        topn=10
    )

    print("Các từ gần nghĩa với", word)

    for item in similar_words:
        print(item)

In [8]:
def document_vector(text, model):

    words = text.split()

    vectors = [
        model.wv[word]
        for word in words
        if word in model.wv
    ]

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)


X = np.array([
    document_vector(text, word2vec_model)
    for text in df["clean_text"].astype(str)
])

y = df["label"].values

print("Kích thước ma trận vector:", X.shape)
print("Kích thước nhãn:", y.shape)

Kích thước ma trận vector: (300, 100)
Kích thước nhãn: (300,)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Số mẫu huấn luyện:", len(X_train))
print("Số mẫu kiểm tra:", len(X_test))

Số mẫu huấn luyện: 240
Số mẫu kiểm tra: 60


In [10]:
word2vec_model.save("word2vec.model")

np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)

np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)

print("Đã lưu mô hình và dữ liệu vector!")

Đã lưu mô hình và dữ liệu vector!
